In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%ls

bert-clinc/  oos-eval/  proiectnlp.ipynb  wandb/


In [5]:
!pip install datasets transformers torch
!pip install nlpaug
!pip install textattack
!pip install openai-whisper
!pip install soundfile
!pip install lemminflect
!pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 44.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 50.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 445.7/445.7 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!git clone https://github.com/clinc/oos-eval

Cloning into 'oos-eval'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 69 (delta 16), reused 16 (delta 16), pack-reused 50 (from 1)
Receiving objects: 100% (69/69), 1.74 MiB | 5.65 MiB/s, done.
Resolving deltas: 100% (35/35), done.


In [ ]:
%cd drive/MyDrive/NLP/

/content/drive/MyDrive/NLP


In [7]:
import nltk
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [8]:
import os
clinc_path = "/content/drive/MyDrive/NLP/oos-eval/data/data_full.json"
print(os.path.exists(clinc_path))

True


In [9]:
import json

with open(clinc_path, "r") as f:
    data = json.load(f)

print(data.keys())


dict_keys(['oos_val', 'val', 'train', 'oos_test', 'test', 'oos_train'])


In [10]:
train_texts = [item[0] for item in data["train"]]
train_labels = [item[1] for item in data["train"]]

test_texts = [item[0] for item in data["test"]]
test_labels = [item[1] for item in data["test"]]

val_texts = [item[0] for item in data["val"]]
val_labels = [item[1] for item in data["val"]]

print(f"Train: {len(train_texts)} examples")
print(f"Test:  {len(test_texts)} examples")
print(f"Val:   {len(val_texts)} examples")

print(train_texts[0], "->", train_labels[0])

Train: 15000 examples
Test:  4500 examples
Val:   3000 examples
what expression would i use to say i love you if i were an italian -> translate


In [11]:
# Get all unique intents
all_labels = list(set(train_labels))
all_labels.sort()

label2id = {label: idx for idx, label in enumerate(all_labels)}
id2label = {idx: label for label, idx in label2id.items()}

num_labels = len(all_labels)
print(f"Number of intents: {num_labels}")

# Convert string labels to integers
train_ids = [label2id[l] for l in train_labels]
test_ids  = [label2id[l] for l in test_labels]
val_ids   = [label2id[l] for l in val_labels]

Number of intents: 150


In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(texts, labels, max_length=64):
    encodings = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return encodings, labels

train_encodings, train_ids = tokenize(train_texts, train_ids)
test_encodings, test_ids   = tokenize(test_texts, test_ids)
val_encodings, val_ids     = tokenize(val_texts, val_ids)

print("Tokenization done")
print("Input shape:", train_encodings["input_ids"].shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Tokenization done
Input shape: torch.Size([15000, 33])


In [ ]:
import torch
from torch.utils.data import Dataset

class IntentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = IntentDataset(train_encodings, train_ids)
test_dataset  = IntentDataset(test_encodings, test_ids)
val_dataset   = IntentDataset(val_encodings, val_ids)

print(f"Train dataset size: {len(train_dataset)}")

Train dataset size: 15000


In [ ]:
from transformers import BertForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/NLP/bert-clinc",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",  
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="/content/logs",
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: alexbrb27 (alexbrb27-universitatea-politehnica-din-bucuresti) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.880631,0.935333
2,2.667500,0.215721,0.967333
3,0.339900,0.175467,0.964667
4,0.076100,0.165672,0.964333
5,0.036300,0.163107,0.966000


TrainOutput(global_step=2345, training_loss=0.6687574504535081, metrics={'train_runtime': 289.8686, 'train_samples_per_second': 258.738, 'train_steps_per_second': 8.09, 'total_flos': 1273564838700000.0, 'train_loss': 0.6687574504535081, 'epoch': 5.0})

In [ ]:
results = trainer.evaluate(test_dataset)
print(f"Clean test accuracy: {results['eval_accuracy']*100:.2f}%")


Clean test accuracy: 96.20%


In [ ]:
# Save model and tokenizer to Drive
save_path = "/content/drive/MyDrive/NLP/bert-clinc/final_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print("Model saved")

Model saved


Adding Noise to the test set

In [6]:
!pip install transformers torch nlpaug nltk gtts openai-whisper -q
!apt-get install -y ffmpeg -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 1.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wikipedia-api 0.14.1 requires click==8.3.2, but you have click 8.1.8 which is incompatible.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [12]:

print(f"Train: {len(train_texts)} | Test: {len(test_texts)} | Val: {len(val_texts)}")
print(f"Intents: {num_labels}")
print(f"Sample: '{test_texts[0]}' -> {test_labels[0]}")

Train: 15000 | Test: 4500 | Val: 3000
Intents: 150
Sample: 'how would you say fly in italian' -> translate


In [13]:
from transformers import BertForSequenceClassification, BertTokenizer
import torch

save_path = "/content/drive/MyDrive/NLP/bert-clinc/final_model"
tokenizer = BertTokenizer.from_pretrained(save_path)
model     = BertForSequenceClassification.from_pretrained(save_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print(f"Model loaded — running on {device}")

Model loaded — running on cpu


In [ ]:
import torch
from torch.utils.data import Dataset
import numpy as np
from sklearn.metrics import accuracy_score

class IntentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

def tokenize(texts):
    return tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=64,
        return_tensors="pt"
    )

def evaluate(texts, label_ids):
    encodings  = tokenize(texts)
    dataset    = IntentDataset(encodings, label_ids)
    loader     = torch.utils.data.DataLoader(dataset, batch_size=64)
    all_preds  = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"]
            outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
            preds          = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return accuracy_score(all_labels, all_preds)

baseline_acc = evaluate(test_texts, test_ids)
print(f"Clean baseline accuracy: {baseline_acc*100:.2f}%")


Clean baseline accuracy: 96.22%


In [ ]:
import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw

# --- Casing ---
def add_casing_noise(texts):
    return [t.upper() for t in texts]

# --- Keyboard typos ---
def add_keyboard_noise(texts, aug_char_p=0.15):
    aug = nac.KeyboardAug(aug_char_p=aug_char_p)
    results = []
    for t in texts:
        try:
            results.append(aug.augment(t)[0])
        except:
            results.append(t)
    return results

# --- Misspellings ---
def add_spelling_noise(texts, aug_p=0.2):
    aug = naw.SpellingAug(aug_p=aug_p)
    results = []
    for t in texts:
        try:
            results.append(aug.augment(t)[0])
        except:
            results.append(t)
    return results

# --- Synonyms ---
def add_synonym_noise(texts, aug_p=0.2):
    aug = naw.SynonymAug(aug_p=aug_p)
    results = []
    for t in texts:
        try:
            results.append(aug.augment(t)[0])
        except:
            results.append(t)
    return results

# --- Abbreviations ---
ABBREV_MAP = {
    "please": "pls", "you": "u", "your": "ur",
    "are": "r", "okay": "ok", "thanks": "thx",
    "tomorrow": "tmrw", "because": "bc",
    "with": "w/", "without": "w/o",
    "information": "info", "application": "app",
    "number": "num", "message": "msg",
    "account": "acct", "department": "dept",
    "appointment": "appt", "maximum": "max",
    "minimum": "min", "approximately": "approx",
}

def add_abbreviation_noise(texts):
    results = []
    for t in texts:
        words     = t.split()
        new_words = [ABBREV_MAP.get(w.lower(), w) for w in words]
        results.append(" ".join(new_words))
    return results

print("Generating casing noise...")
noisy_casing = add_casing_noise(test_texts)

print("Generating keyboard noise...")
noisy_keyboard = add_keyboard_noise(test_texts)

print("Generating spelling noise...")
noisy_spelling = add_spelling_noise(test_texts)

print("Generating synonym noise...")
noisy_synonyms = add_synonym_noise(test_texts)

print("Generating abbreviation noise...")
noisy_abbrev = add_abbreviation_noise(test_texts)

print("All noise variants ready")

for name, noisy in [("casing", noisy_casing), ("keyboard", noisy_keyboard),
                     ("spelling", noisy_spelling), ("synonyms", noisy_synonyms),
                     ("abbreviations", noisy_abbrev)]:
    print(f"{name}: '{test_texts[0]}' -> '{noisy[0]}'")

Generating casing noise...
Generating keyboard noise...
Generating spelling noise...
Generating synonym noise...


Datele de ieșire de afișat au fost trunchiate la ultimele 5000 linii.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]   

Generating abbreviation noise...
All noise variants ready
casing: 'how would you say fly in italian' -> 'HOW WOULD YOU SAY FLY IN ITALIAN'
keyboard: 'how would you say fly in italian' -> 'how wou/d you say fly in itzl8an'
spelling: 'how would you say fly in italian' -> 'how would you say fligt in Italy'
synonyms: 'how would you say fly in italian' -> 'how would you say fly in italian'
abbreviations: 'how would you say fly in italian' -> 'how would u say fly in italian'


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger t

In [16]:
noise_variants = {
    "original":      test_texts,
    "casing":        noisy_casing,
    "keyboard":      noisy_keyboard,
    "spelling":      noisy_spelling,
    "synonyms":      noisy_synonyms,
    "abbreviations": noisy_abbrev,
}

results = {}
for noise_type, texts in noise_variants.items():
    acc = evaluate(texts, test_ids)
    results[noise_type] = acc
    print(f"{noise_type:<20} accuracy: {acc*100:.2f}%")

# PDR and ERM
print("\n--- Degradation Table ---")
print(f"{'Noise Type':<20} {'Accuracy':>10} {'PDR':>10} {'ERM':>10}")
print("-" * 55)
clean_acc = results["original"]
for noise_type, acc in results.items():
    if noise_type == "original":
        print(f"{noise_type:<20} {acc*100:>9.2f}%  {'—':>10} {'—':>10}")
    else:
        pdr = (clean_acc - acc) / clean_acc * 100
        erm = (1 - acc) / (1 - clean_acc)
        print(f"{noise_type:<20} {acc*100:>9.2f}% {pdr:>9.2f}% {erm:>10.2f}x")

original             accuracy: 96.22%
casing               accuracy: 96.22%
keyboard             accuracy: 43.31%
spelling             accuracy: 82.84%
synonyms             accuracy: 96.22%
abbreviations        accuracy: 95.31%

--- Degradation Table ---
Noise Type             Accuracy        PDR        ERM
-------------------------------------------------------
original                 96.22%           —          —
casing                   96.22%      0.00%       1.00x
keyboard                 43.31%     54.99%      15.01x
spelling                 82.84%     13.90%       4.54x
synonyms                 96.22%      0.00%       1.00x
abbreviations            95.31%      0.95%       1.24x


In [ ]:
import json


noisy_data = {
    "original":      test_texts,
    "casing":        noisy_casing,
    "keyboard":      noisy_keyboard,
    "spelling":      noisy_spelling,
    "synonyms":      noisy_synonyms,
    "abbreviations": noisy_abbrev,
}
with open("/content/drive/MyDrive/NLP/bert-clinc/noisy_test_sets.json", "w") as f:
    json.dump(noisy_data, f)

with open("/content/drive/MyDrive//NLP/bert-clinc/results.json", "w") as f:
    json.dump(results, f)

print("Saved to Drive")

Saved to Drive


In [ ]:
noise_type = "keyboard" 

print(f"{'ORIGINAL':<45} {'NOISY'}")
print("-" * 90)
for i in range(20):
    print(f"{test_texts[i]:<45} {noisy_data[noise_type][i]}")

ORIGINAL                                      NOISY
------------------------------------------------------------------------------------------
how would you say fly in italian              how wou/d you say fly in itzl8an
what's the spanish word for pasta             whzt ' s the spznisg wo4d for pasta
how would they say butter in zambia           how wo^ld theh say butrer in zambia
how do you say fast in spanish                how do you say fZst in cpahish
what's the word for trees in norway           wha6 ' s the 2ord for treFs in norway
how does one say wonderful in german          how roes one say wonwFrful in germAn
how do they say tacos in mexico               how do tbey say taVos in mexick
how would one say cruiser in china            how woild one say cEuisef in Vhina
what's the french word you use for potato     whSt ' s the f5ench word you use for po6ato
what would the word for grass be in finland   whaH wou,d the word for grass be in finianv
how do you say please in french

In [ ]:

from transformers import BertForSequenceClassification, BertTokenizer

tokenizer_cased = BertTokenizer.from_pretrained("bert-base-cased")
model_cased = BertForSequenceClassification.from_pretrained(
    "bert-base-cased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
